In [1]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"


In [2]:
#____________________________ part a ______________________________#
from __future__ import unicode_literals 
from hazm import * 
import zipfile
import io
import pandas as pd
import re
import nltk
import os
from nltk.corpus import stopwords
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# Opening the zip file in read mode
with zipfile.ZipFile('sentiment dataset.zip', 'r') as myzip:
    csv_file = myzip.open('Snappfood - Sentiment Analysis.csv')
    data = pd.read_csv(io.BytesIO(csv_file.read()), delimiter='\t', error_bad_lines=False)

data.head()
#data.isnull()

for attr in ['Unnamed: 0', 'label', 'label_id']:
    print(data[attr].value_counts())

#other_data = data[data['label'].isin(['0','1'])]
#other_data.head()

data = data[~data['label'].isin(['0', '1'])]

for attr in ['Unnamed: 0', 'label', 'label_id']:
    print(data[attr].value_counts())
    
data.drop(columns=['Unnamed: 0','label_id'],inplace=True)
data

def remove_tags(string):
    removelist = ""
    result = re.sub('<.*?>', '', string)   #remove HTML tags
    result = re.sub('http\S+', '', result)  #remove URLs
    result = re.sub(r'[^\w\s'+removelist+']', ' ', result, flags=re.UNICODE)   #remove non-alphanumeric characters 
    result = result.lower()
    return result

#nltk.download('stopwords')
#data_dir = os.path.expanduser('~/nltk_data')
#persian_stopwords_path = os.path.join(data_dir, 'corpora', 'stopwords', 'persian')
stop_words = ['/','؟','.','«','»','>','<',':','؛','!','٪','،',')','(','ـ','-','=','+','*']
data['comment'] = data['comment'].apply(lambda x: remove_tags(x))
data['comment'] = data['comment'].apply(lambda x: ' '.join([word for word in x.split() if word not in stop_words]))
data.sample(10)

lemmatizer = Lemmatizer()

def lemmatize_text(text):
    st = ""
    for w in word_tokenize(text):
        st = st + lemmatizer.lemmatize(w) + " "
    return st

data['comment'] = data['comment'].apply(lemmatize_text)
data.sample(10)

reviews = data['comment'].values
labels = data['label'].values
encoder = LabelEncoder()
encoded_labels = encoder.fit_transform(labels)

train_sentences, test_sentences, train_labels, test_labels = train_test_split(reviews, encoded_labels, test_size=0.2, stratify = encoded_labels)


/home/nima/anaconda3/envs/sklearnenv5/lib/python3.7/site-packages/IPython/core/interactiveshell.py:3457: FutureWarning: The error_bad_lines argument has been deprecated and will be removed in a future version.


  exec(code_obj, self.user_global_ns, self.user_ns)


,Unnamed: 0,comment,label,label_id
0,NaN,واقعا حیف وقت که بنویسم سرویس دهیتون شده افتضاح,SAD,1.0
1,NaN,قرار بود ۱ ساعته برسه ولی نیم ساعت زودتر از مو...,HAPPY,0.0
2,NaN,قیمت این مدل اصلا با کیفیتش سازگاری نداره، فقط...,SAD,1.0
3,NaN,عالللی بود همه چه درست و به اندازه و کیفیت خوب...,HAPPY,0.0
4,NaN,شیرینی وانیلی فقط یک مدل بود.,HAPPY,0.0


 هزار پول ساندویچ شد، ولی بیشترش قارچ بود                                                                                                                                                                                                                1
عدد از بستنی‌ها واقعا له شده بود و قابل استفاده نبود                                                                                                                                                                                                     1
 گرم سالاد شیرازی گذاشته بودن که همه میدونیم مواد تشکیل دهنده ش گوجه و خیار و پیازه، اگه حساب کنیم قیمت تمام شده ش ۲ تومنم نمیشه، پس چرا ۶۰۰۰ تومن قیمت این سالاده؟ یا زیتون پرورده ش هم همینطور، اصلا انصاف نیس. در کل قیمت با حجم اصلا متناسب نبود.    1
 تا نون سفارش دادم توی ۲ تا بسته تحویل شد یکیش کاملا نون هاش بیات و مونده بود،                                                                                                                                                                         

,comment,label
0,واقعا حیف وقت که بنویسم سرویس دهیتون شده افتضاح,SAD
1,قرار بود ۱ ساعته برسه ولی نیم ساعت زودتر از مو...,HAPPY
2,قیمت این مدل اصلا با کیفیتش سازگاری نداره، فقط...,SAD
3,عالللی بود همه چه درست و به اندازه و کیفیت خوب...,HAPPY
4,شیرینی وانیلی فقط یک مدل بود.,HAPPY
...,...,...
69995,سلام من به فاکتور غذاهایی که سفارش میدم احتیاج...,SAD
69996,سایز پیتزا نسبت به سفارشاتی که قبلا گذشتم کم ش...,SAD
69997,من قارچ اضافه رو اضافه کرده بودم بودم اما اگر ...,HAPPY
69998,همرو بعد ۲ساعت تاخیر اشتباه آوردن پولشم رفت رو...,SAD


,comment,label
63003,کیفیت سالاد مثل همیشه عالی بود منتهی یه انتقاد...,HAPPY
29420,کاملا سریع به دستم رسید,HAPPY
51715,کاملا ساندویچ تلخ و خیلی چرب بود خیلی بد بود,SAD
69844,با سلام متاسفانه دوغ ارسالی بدون گاز هست و بعد...,SAD
54244,یه مقدار عملکردتون توی اسنپ فود ضعیفه خب وقتی ...,HAPPY
5855,برخورد پیک خیلی خوب بود به نسبت هزینه ای که بر...,SAD
43940,متاسفانه بر خلاف همیشه غذا سرد تحویل شد و خیلی...,SAD
35027,متاسفانه مایع نظافت روی مواد غذایی و دستمال کا...,SAD
30072,مرررسی غذا خیلی خوب بود و از اینکه گرم بدستم ر...,HAPPY
25524,افتضاح نان ساندویچ و مرغ مونده بود خشک و سفت و...,SAD


,comment,label
54435,به توضیحات درخواست توجه نکردین,SAD
21342,پیتزا آمریکایی خوب و خوشمزه بود#باش سیب زمین غ...,HAPPY
46678,سلام عزیز عکس بود#باش و شیرینی کم داشت شکلات ب...,HAPPY
45949,چقد تمیزو مرتب شیرینی رو میفرستن چقد جعبه شون ...,HAPPY
38193,با تاخیر فراوان و قیمت خییییییلی بالا اینکه می...,SAD
27815,پنکیک خیلی کره بود#باش و خفن,HAPPY
27649,سلام خیلی ممنون از تحویل سریع چیپس کرانچیپس رو...,HAPPY
66443,در بسته آب آشامیدن یک عدد دماوند جایگزین شد#شو,SAD
27517,مرغ کاملا خام و قرمز بو بد مرغ کل پک جعبه را گ...,SAD
19543,از هر لحاظ عالی بود#باش,HAPPY


In [3]:
#____________________________ part b ______________________________#
from collections import defaultdict
import math
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score
import numpy as np 

# Creating a CountVectorizer object with a maximum of 3000 features
vec = CountVectorizer(max_features = 3000)

# Transforming the training sentences into a document-term matrix
X = vec.fit_transform(train_sentences)

# Obtaining the feature names from the CountVectorizer object
vocab = vec.get_feature_names_out()

# Converting the sparse matrix to a dense array
X = X.toarray()

# Initializing a dictionary to keep track of word counts for each label
word_counts = {}
for l in range(2):
    word_counts[l] = defaultdict(lambda: 0)

# Iterating through each training sentence and counting the occurrence of each word
for i in range(X.shape[0]):
    l = train_labels[i]
    for j in range(len(vocab)):
        word_counts[l][vocab[j]] += X[i][j]

# Function to perform Laplace smoothing for a given word and label
def laplace_smoothing(n_label_items, vocab, word_counts, word, text_label):
    a = word_counts[text_label][word] + 1
    b = n_label_items[text_label] + len(vocab)
    return math.log(a/b)

# Function to group training data by label
def group_by_label(x, y, labels):
    data = {}
    for l in labels:
        data[l] = x[np.where(y == l)]
    return data


In [4]:
#____________________________ part c ______________________________#

# Function to fit the Naive Bayes classifier to the training data
def fit(x, y, labels):
    n_label_items = {}
    log_label_priors = {}
    n = len(x)
    grouped_data = group_by_label(x, y, labels)
    for l, data in grouped_data.items():
        n_label_items[l] = len(data)
        log_label_priors[l] = math.log(n_label_items[l] / n)
    return n_label_items, log_label_priors

# Function to make predictions using the Naive Bayes classifier
def predict(n_label_items, vocab, word_counts, log_label_priors, labels, x):
    result = []
    for text in x:
        label_scores = {l: log_label_priors[l] for l in labels}
        words = set(word_tokenize(text))
        for word in words:
            if word not in vocab: continue
            for l in labels:
                log_w_given_l = laplace_smoothing(n_label_items, vocab, word_counts, word, l)
                label_scores[l] += log_w_given_l
        result.append(max(label_scores, key=label_scores.get))
    return result

# Defining the possible labels for the classifier
labels = [0,1]

# Fitting the Naive Bayes classifier to the training data
n_label_items, log_label_priors = fit(train_sentences,train_labels,labels)

# Using the trained classifier to make predictions on the test data
pred = predict(n_label_items, vocab, word_counts, log_label_priors, labels, test_sentences)

# Calculating the accuracy of the classifier on the test data
print("Accuracy of prediction on test set : ", accuracy_score(test_labels,pred))


Accuracy of prediction on test set :  0.7822394933793898
